# TRENDYv10 extraction at flux-tower/tree-ring sites

This notebook produces the annual site-grid-cell CSV files used immediately by `Figure1_code_clean.Rmd`.

## Run

Create and activate the recorded environment, then run from the repository's `code/` directory:

```bash
conda env create -f environment_trendy.yml
conda activate source-sink-trendy
jupyter nbconvert --to notebook --execute TRENDY_extraction_flux_locations_clean.ipynb \
  --inplace --ExecutePreprocessor.timeout=-1
```

When running elsewhere, set `SOURCE_SINK_ROOT=/path/to/source_sink_dynamics`.
For a bounded run, set comma-separated `TRENDY_MODELS` or `TRENDY_SCENARIOS`; for example:

```bash
TRENDY_MODELS=CABLE-POP TRENDY_SCENARIOS=S2 jupyter nbconvert --to notebook \
  --execute TRENDY_extraction_flux_locations_clean.ipynb --inplace
```

Inputs follow `data/TRENDYv10/downloads/<MODEL>/<MODEL>_<SCENARIO>_<VARIABLE>.nc`.
Outputs follow `results/TRENDYv10/31_site_weighted/<MODEL>/`.


## 1. Configuration and dependencies

Edit environment variables rather than machine-specific paths.


In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import netCDF4 as nc
import numpy as np
import pandas as pd
import xarray as xr

DEFAULT_MODELS = (
    "ISBA-CTRIP",
    "CABLE-POP",
    "CLM5.0",
    "LPJ-GUESS",
    "LPX-Bern",
    "ORCHIDEE",
    "ORCHIDEEv3",
    "CLASSIC-N",
    "CLASSIC",
)
DEFAULT_SCENARIOS = ("S0", "S1", "S2")
VARIABLES = ("gpp", "cWood")


def env_list(name: str, default: tuple[str, ...]) -> tuple[str, ...]:
    value = os.getenv(name, "").strip()
    return tuple(item.strip() for item in value.split(",") if item.strip()) if value else default


# Run from code/, or set SOURCE_SINK_ROOT explicitly.
PROJECT_ROOT = Path(os.getenv("SOURCE_SINK_ROOT", "..")).expanduser().resolve()
INPUT_DIR = Path(
    os.getenv(
        "TRENDY_INPUT_DIR",
        PROJECT_ROOT / "data" / "TRENDYv10" / "downloads",
    )
).expanduser().resolve()
CABON_DATA_DIR = Path(
    os.getenv("CABON_DATA_DIR", PROJECT_ROOT / "data" / "cabon")
).expanduser().resolve()
SITE_INFO_FILE = CABON_DATA_DIR / "Cabonetal_site_info.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "TRENDYv10" / "31_site_weighted"

MODELS = env_list("TRENDY_MODELS", DEFAULT_MODELS)
SCENARIOS = env_list("TRENDY_SCENARIOS", DEFAULT_SCENARIOS)

unknown_models = sorted(set(MODELS) - set(DEFAULT_MODELS))
unknown_scenarios = sorted(set(SCENARIOS) - set(DEFAULT_SCENARIOS))
if unknown_models:
    raise ValueError(f"Unknown models in TRENDY_MODELS: {unknown_models}")
if unknown_scenarios:
    raise ValueError(f"Unknown scenarios in TRENDY_SCENARIOS: {unknown_scenarios}")
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"TRENDY input directory not found: {INPUT_DIR}")
if not SITE_INFO_FILE.is_file():
    raise FileNotFoundError(f"Site metadata not found: {SITE_INFO_FILE}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"TRENDY inputs: {INPUT_DIR}")
print(f"Cabon inputs: {CABON_DATA_DIR}")
print(f"Models: {', '.join(MODELS)}")
print(f"Scenarios: {', '.join(SCENARIOS)}")
print(f"Variables: {', '.join(VARIABLES)}")


## 2. Reusable extraction functions

Coordinates and calendars are standardized before site selection.


In [ ]:
COORDINATE_ALIASES = {
    "lon": ("lon", "longitude", "Longitude", "LONGITUDE", "X", "x", "lon_FULL"),
    "lat": ("lat", "latitude", "Latitude", "LATITUDE", "Y", "y", "lat_FULL"),
    "time": ("time", "time_counter", "Time", "TIME"),
}


def input_path(model: str, scenario: str, variable: str) -> Path:
    return INPUT_DIR / model / f"{model}_{scenario}_{variable}.nc"


def output_path(model: str, scenario: str, variable: str) -> Path:
    return (
        OUTPUT_DIR
        / model
        / f"{variable}_{scenario}_31_site_weighted_yearly_mean.csv"
    )


def rename_coordinates(dataset: xr.Dataset) -> xr.Dataset:
    rename_map = {}
    for standard_name, aliases in COORDINATE_ALIASES.items():
        match = next(
            (name for name in aliases if name in dataset.coords or name in dataset.variables),
            None,
        )
        if match is None:
            raise ValueError(
                f"Could not identify {standard_name!r} coordinate. "
                f"Available coordinates: {list(dataset.coords)}"
            )
        if match != standard_name:
            rename_map[match] = standard_name
    return dataset.rename(rename_map) if rename_map else dataset


def decode_time_coordinate(dataset: xr.Dataset) -> xr.Dataset:
    time = dataset["time"]
    units = time.attrs.get("units")
    calendar = time.attrs.get("calendar", "standard")

    if not units:
        raise ValueError("The NetCDF time coordinate has no 'units' attribute.")

    decode_units = units
    if units.startswith("months since"):
        calendar = "360_day"
    elif calendar in {"noleap", "365_day"} and units.startswith("years since"):
        decode_units = units.replace("years since", "common_years since", 1)

    dates = nc.num2date(
        time.values,
        units=decode_units,
        calendar=calendar,
        only_use_cftime_datetimes=True,
    )
    return dataset.assign_coords(time=("time", dates))


def normalize_longitude(dataset: xr.Dataset) -> xr.Dataset:
    longitude = ((dataset["lon"] + 180) % 360) - 180
    return dataset.assign_coords(lon=longitude).sortby("lon")


def open_trendy_dataset(path: Path) -> xr.Dataset:
    if not path.is_file():
        raise FileNotFoundError(f"TRENDY file not found: {path}")

    dataset = xr.open_dataset(path, decode_times=False)
    try:
        dataset = rename_coordinates(dataset)
        dataset = decode_time_coordinate(dataset)
        dataset = normalize_longitude(dataset)
    except Exception:
        dataset.close()
        raise
    return dataset


def read_site_metadata() -> pd.DataFrame:
    sites = pd.read_csv(SITE_INFO_FILE)
    required_columns = {"Site", "Lat.", "Lon.", "On-site RW"}
    missing_columns = sorted(required_columns - set(sites.columns))
    if missing_columns:
        raise ValueError(
            f"Site metadata is missing columns: {', '.join(missing_columns)}"
        )

    sites = (
        sites.loc[sites["On-site RW"].eq(True), ["Site", "Lat.", "Lon."]]
        .rename(
            columns={
                "Site": "SITE_ID",
                "Lat.": "site_lat",
                "Lon.": "site_lon",
            }
        )
        .drop_duplicates("SITE_ID")
        .reset_index(drop=True)
    )
    sites["site_lon"] = ((sites["site_lon"] + 180) % 360) - 180

    if sites.empty:
        raise ValueError("No on-site ring-width locations were found.")
    return sites


def map_sites_to_grid(dataset: xr.Dataset, sites: pd.DataFrame) -> pd.DataFrame:
    latitudes = np.asarray(dataset["lat"].values)
    longitudes = np.asarray(dataset["lon"].values)

    lat_indices = np.abs(
        latitudes[:, np.newaxis] - sites["site_lat"].to_numpy()
    ).argmin(axis=0)
    lon_indices = np.abs(
        longitudes[:, np.newaxis] - sites["site_lon"].to_numpy()
    ).argmin(axis=0)

    mapping = sites.copy()
    mapping["lat_index"] = lat_indices
    mapping["lon_index"] = lon_indices
    mapping["lat"] = latitudes[lat_indices]
    mapping["lon"] = longitudes[lon_indices]
    return mapping


def validate_grid(dataset: xr.Dataset, reference: xr.Dataset, path: Path) -> None:
    same_latitude = np.array_equal(dataset["lat"].values, reference["lat"].values)
    same_longitude = np.array_equal(dataset["lon"].values, reference["lon"].values)
    if not (same_latitude and same_longitude):
        raise ValueError(f"Grid differs from the model template: {path}")


def extract_annual_grid_cells(
    dataset: xr.Dataset,
    variable: str,
    grid_mapping: pd.DataFrame,
) -> pd.DataFrame:
    if variable not in dataset:
        raise KeyError(
            f"Variable {variable!r} is absent. Available variables: "
            f"{list(dataset.data_vars)}"
        )

    values = dataset[variable]
    extra_dimensions = set(values.dims) - {"time", "lat", "lon"}
    non_singleton = {
        dimension: values.sizes[dimension]
        for dimension in extra_dimensions
        if values.sizes[dimension] != 1
    }
    if non_singleton:
        raise ValueError(
            f"{variable!r} has unsupported extra dimensions: {non_singleton}"
        )
    if extra_dimensions:
        values = values.squeeze(tuple(extra_dimensions), drop=True)

    # Extract only unique site-overlapping grid cells. This avoids materializing
    # an entire global grid filled mostly with NA values.
    locations = (
        grid_mapping[["lat_index", "lon_index", "lat", "lon"]]
        .drop_duplicates(["lat_index", "lon_index"])
        .reset_index(drop=True)
    )
    lat_indexer = xr.DataArray(locations["lat_index"].to_numpy(), dims="location")
    lon_indexer = xr.DataArray(locations["lon_index"].to_numpy(), dims="location")
    selected = values.isel(lat=lat_indexer, lon=lon_indexer)

    annual = selected.groupby("time.year").mean(dim="time", skipna=True)
    annual_frame = annual.to_dataframe(name=variable).reset_index()

    # Use coordinates from the mapping to preserve exact values shared with the
    # model's site-info file.
    annual_frame = (
        annual_frame.drop(columns=["lat", "lon"], errors="ignore")
        .merge(
            locations.reset_index().rename(columns={"index": "location"}),
            on="location",
            how="left",
            validate="many_to_one",
        )
        [["year", "lat", "lon", variable]]
        .dropna(subset=[variable])
        .sort_values(["lat", "lon", "year"])
        .reset_index(drop=True)
    )
    return annual_frame


## 3. Extract annual values

Only unique grid cells overlapping the observational sites are retained.


In [ ]:
sites = read_site_metadata()
manifest_rows = []
started_at = time.time()

for model in MODELS:
    model_output_dir = OUTPUT_DIR / model
    model_output_dir.mkdir(parents=True, exist_ok=True)

    template_path = input_path(model, "S2", "gpp")
    print(f"\n[{model}] Reading template grid: {template_path.name}")

    with open_trendy_dataset(template_path) as template:
        grid_mapping = map_sites_to_grid(template, sites)
        reference_latitude = template[["lat"]].load()
        reference_longitude = template[["lon"]].load()
        reference_grid = xr.merge([reference_latitude, reference_longitude])

        site_output = model_output_dir / f"{model}_site_info.csv"
        (
            grid_mapping[["lat", "lon", "SITE_ID", "site_lat", "site_lon"]]
            .sort_values("SITE_ID")
            .to_csv(site_output, index=False)
        )
        print(f"[{model}] Wrote {site_output.name} ({len(grid_mapping)} sites)")

        for scenario in SCENARIOS:
            for variable in VARIABLES:
                source = input_path(model, scenario, variable)
                destination = output_path(model, scenario, variable)

                with open_trendy_dataset(source) as dataset:
                    validate_grid(dataset, reference_grid, source)
                    annual = extract_annual_grid_cells(
                        dataset,
                        variable,
                        grid_mapping,
                    )

                annual.to_csv(destination, index=False)
                manifest_rows.append(
                    {
                        "model": model,
                        "scenario": scenario,
                        "variable": variable,
                        "rows": len(annual),
                        "output": str(destination.relative_to(PROJECT_ROOT)),
                    }
                )
                print(
                    f"[{model} {scenario} {variable}] "
                    f"Wrote {len(annual):,} rows to {destination.name}"
                )

elapsed_minutes = (time.time() - started_at) / 60
manifest = pd.DataFrame(manifest_rows)
print(f"\nCompleted {len(manifest)} files in {elapsed_minutes:.1f} minutes.")
manifest


## 4. Validate the Figure 1 output contract


In [ ]:
expected_files = [
    output_path(model, scenario, variable)
    for model in MODELS
    for scenario in SCENARIOS
    for variable in VARIABLES
]
expected_site_files = [
    OUTPUT_DIR / model / f"{model}_site_info.csv"
    for model in MODELS
]
missing_outputs = [
    path for path in expected_files + expected_site_files if not path.is_file()
]
if missing_outputs:
    raise RuntimeError(
        "Extraction completed with missing outputs:\n"
        + "\n".join(f"- {path}" for path in missing_outputs)
    )

# Verify the output schema consumed by Figure1_code_clean.Rmd.
for path in expected_files:
    variable = path.name.split("_", 1)[0]
    columns = set(pd.read_csv(path, nrows=1).columns)
    required = {"year", "lat", "lon", variable}
    if not required.issubset(columns):
        raise RuntimeError(f"Unexpected columns in {path}: {sorted(columns)}")

for path in expected_site_files:
    columns = set(pd.read_csv(path, nrows=1).columns)
    if not {"lat", "lon", "SITE_ID"}.issubset(columns):
        raise RuntimeError(f"Unexpected site-info columns in {path}: {sorted(columns)}")

print(
    f"Validated {len(expected_files)} annual files and "
    f"{len(expected_site_files)} site-info files."
)
